In [1]:
pip install boto3 botocore

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 1.7 MB/s  0:00:08 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [boto3]32m1/4 [botocore]
Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install warcio trafilatura beautifulsoup4 lxml pandas pyarrow tqdm


  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 25.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 27.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 14.6 MB/s  0:00:00
Using cached soupsieve-2.8.3-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [trafilatura] [trafilatura]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import datetime as dt
import gzip, io
import re
import requests

BASE = "https://data.commoncrawl.org/"
TS_RE = re.compile(r"CC-NEWS-(\d{14})-\d{5}\.warc\.gz$")

def fetch_warc_paths(year: int, month: int) -> list[str]:
    url = f"{BASE}crawl-data/CC-NEWS/{year:04d}/{month:02d}/warc.paths.gz"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as f:
        return [line.decode("utf-8").strip() for line in f if line.strip()]

def prev_month(y: int, m: int) -> tuple[int, int]:
    return (y - 1, 12) if m == 1 else (y, m - 1)

def latest_available_month(max_lookback_months: int = 24) -> tuple[int, int, list[str]]:
    today = dt.date.today()
    y, m = today.year, today.month
    for _ in range(max_lookback_months):
        try:
            paths = fetch_warc_paths(y, m)
            return y, m, paths
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                y, m = prev_month(y, m)
                continue
            raise
    raise RuntimeError("Aucun warc.paths.gz trouvé sur la fenêtre de lookback")

def sort_key(path: str) -> str:
    # tri simple : le timestamp est dans le nom de fichier -> tri lexicographique OK
    # ex: CC-NEWS-20230721173757-00004.warc.gz
    return path

y, m, paths = latest_available_month()
paths_sorted = sorted(paths, key=sort_key)

print(f"Latest month found: {y:04d}/{m:02d} with {len(paths_sorted)} WARC files")
print("Last 5 WARC paths:")
for p in paths_sorted[-5:]:
    print(" -", p)

# Exemple: URL HTTPS du dernier WARC
last_warc_url = BASE + paths_sorted[-1]
print("Last WARC URL:", last_warc_url)


Latest month found: 2026/02 with 159 WARC files
Last 5 WARC paths:
 - crawl-data/CC-NEWS/2026/02/CC-NEWS-20260211224450-06781.warc.gz
 - crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212003559-06782.warc.gz
 - crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212025433-06783.warc.gz
 - crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212051501-06784.warc.gz
 - crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212071821-06785.warc.gz
Last WARC URL: https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212071821-06785.warc.gz


In [4]:
import os
import datetime as dt
import gzip, io
import requests

BASE = "https://data.commoncrawl.org/"

def fetch_warc_paths(year: int, month: int) -> list[str]:
    url = f"{BASE}crawl-data/CC-NEWS/{year:04d}/{month:02d}/warc.paths.gz"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with gzip.GzipFile(fileobj=io.BytesIO(r.content)) as f:
        return [line.decode("utf-8").strip() for line in f if line.strip()]

def prev_month(y: int, m: int) -> tuple[int, int]:
    return (y - 1, 12) if m == 1 else (y, m - 1)

def latest_available_month(max_lookback_months: int = 24) -> tuple[int, int, list[str]]:
    today = dt.date.today()
    y, m = today.year, today.month
    for _ in range(max_lookback_months):
        try:
            paths = fetch_warc_paths(y, m)
            return y, m, paths
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code == 404:
                y, m = prev_month(y, m)
                continue
            raise
    raise RuntimeError("Aucun warc.paths.gz trouvé sur la fenêtre de lookback")

def download_file(url: str, out_path: str, chunk_size: int = 1024 * 1024) -> None:
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Si le fichier existe déjà et a une taille > 0, on skip (simple)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"SKIP exists: {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)")
        return

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", "0")) or None
        downloaded = 0

        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                f.write(chunk)
                downloaded += len(chunk)

        if total:
            print(f"OK  {out_path} ({downloaded/1e6:.1f} MB / {total/1e6:.1f} MB)")
        else:
            print(f"OK  {out_path} ({downloaded/1e6:.1f} MB)")

def main() -> None:
    y, m, paths = latest_available_month()
    paths_sorted = sorted(paths)  # tri lexicographique OK

    last5 = paths_sorted[-5:]
    print(f"Latest month found: {y:04d}/{m:02d} -> downloading last {len(last5)} WARC files")

    out_dir = os.path.join(os.getcwd(), "ccnews_warc_last5", f"{y:04d}_{m:02d}")
    for p in last5:
        url = BASE + p
        filename = os.path.basename(p)  # ex: CC-NEWS-...warc.gz
        out_path = os.path.join(out_dir, filename)
        print("GET", url)
        download_file(url, out_path)

if __name__ == "__main__":
    main()


Latest month found: 2026/02 -> downloading last 5 WARC files
GET https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/02/CC-NEWS-20260211224450-06781.warc.gz
OK  /Users/yohan/Documents/3Aschool/projetPFE/ccnews_warc_last5/2026_02/CC-NEWS-20260211224450-06781.warc.gz (1072.7 MB / 1072.7 MB)
GET https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212003559-06782.warc.gz
OK  /Users/yohan/Documents/3Aschool/projetPFE/ccnews_warc_last5/2026_02/CC-NEWS-20260212003559-06782.warc.gz (1072.7 MB / 1072.7 MB)
GET https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212025433-06783.warc.gz
OK  /Users/yohan/Documents/3Aschool/projetPFE/ccnews_warc_last5/2026_02/CC-NEWS-20260212025433-06783.warc.gz (1072.7 MB / 1072.7 MB)
GET https://data.commoncrawl.org/crawl-data/CC-NEWS/2026/02/CC-NEWS-20260212051501-06784.warc.gz
OK  /Users/yohan/Documents/3Aschool/projetPFE/ccnews_warc_last5/2026_02/CC-NEWS-20260212051501-06784.warc.gz (1072.7 MB / 1072.7 MB)
GET https://data.co

In [7]:
import os
import io
import re
import gzip
import json
from urllib.parse import urlparse
from datetime import datetime
from typing import Optional, Dict, Any, Iterable, List

import requests
from warcio.archiveiterator import ArchiveIterator
from tqdm import tqdm

import trafilatura
from bs4 import BeautifulSoup


# -------------------------
# Utils dates / parsing
# -------------------------

def safe_dt(s: Optional[str]) -> Optional[str]:
    """Convertit une date ISO-ish en 'YYYY-MM-DD HH:MM:SS' si possible."""
    if not s:
        return None
    try:
        s2 = s.strip().replace("Z", "")
        dt = datetime.fromisoformat(s2)
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return None


def pick(extracted: Any, key: str, default=None):
    """
    Lit un champ depuis dict ou objet Document (trafilatura).
    Compatible avec plusieurs versions de trafilatura.
    """
    if extracted is None:
        return default

    if isinstance(extracted, dict):
        return extracted.get(key, default)

    if hasattr(extracted, "as_dict"):
        try:
            d = extracted.as_dict()
            if isinstance(d, dict):
                return d.get(key, default)
        except Exception:
            pass

    return getattr(extracted, key, default)


def normalize_date_value(v: Any) -> Optional[str]:
    """Accepte str ou datetime, renvoie 'YYYY-MM-DD HH:MM:SS' ou None."""
    if v is None:
        return None
    if hasattr(v, "strftime"):  # datetime-like
        try:
            return v.strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return None
    if isinstance(v, str):
        return safe_dt(v)
    return None


# -------------------------
# Fallback meta tags
# -------------------------

def meta_fallback(html: str) -> Dict[str, Optional[str]]:
    """Fallback simple via meta tags og:*, twitter:* et <title>."""
    soup = BeautifulSoup(html, "lxml")

    def get_meta(*keys: str) -> Optional[str]:
        for k in keys:
            tag = soup.find("meta", attrs={"property": k}) or soup.find("meta", attrs={"name": k})
            if tag and tag.get("content"):
                return tag["content"].strip()
        return None

    title = get_meta("og:title", "twitter:title")
    desc = get_meta("og:description", "twitter:description", "description")
    img = get_meta("og:image", "twitter:image")

    if not title:
        t = soup.find("title")
        title = t.get_text(strip=True) if t else None

    return {"title": title, "description": desc, "image_url": img}


# -------------------------
# Normalisation d'un record
# -------------------------

def normalize_record(url: str, warc_date: Optional[str], html: str) -> Optional[Dict[str, Any]]:
    """
    Construit un dict {date, description, domain, image_url, text, title, url}.
    Date = publication si trouvée, sinon WARC-Date (crawl).
    """
    parsed = urlparse(url)
    domain = parsed.netloc

    extracted = trafilatura.bare_extraction(
        html,
        url=url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    if not extracted:
        return None

    text = (pick(extracted, "text", "") or "").strip()
    title = pick(extracted, "title", None)
    description = pick(extracted, "description", None)
    image_url = pick(extracted, "image", None)
    pub_date = pick(extracted, "date", None)

    date_norm = normalize_date_value(pub_date) or safe_dt(warc_date)

    # Fallback meta si manquant
    if not title or not description or not image_url:
        fb = meta_fallback(html)
        title = title or fb["title"]
        description = description or fb["description"]
        image_url = image_url or fb["image_url"]

    # Fallback description: début du texte
    if not description and text:
        description = (text[:280] + "…") if len(text) > 280 else text

    # Filtre anti-bruit minimal
    if len(text) < 200:
        return None
    if not title:
        return None

    return {
        "date": date_norm,
        "description": description,
        "domain": domain,
        "image_url": image_url,
        "text": text,
        "title": title,
        "url": url,
    }


# -------------------------
# Lecture WARC
# -------------------------

def iter_warc_records(warc_gz_path: str) -> Iterable[Dict[str, Any]]:
    """Yield records normalisés depuis un fichier .warc.gz."""
    with gzip.open(warc_gz_path, "rb") as stream:
        for rec in ArchiveIterator(stream):
            if rec.rec_type != "response":
                continue

            url = rec.rec_headers.get_header("WARC-Target-URI")
            warc_date = rec.rec_headers.get_header("WARC-Date")  # date crawl

            if not url:
                continue

            # Filtrer HTML
            ctype = ""
            if rec.http_headers:
                ctype = rec.http_headers.get_header("Content-Type") or ""
            if "text/html" not in ctype.lower():
                continue

            try:
                raw = rec.content_stream().read()
                html = raw.decode("utf-8", errors="ignore")
            except Exception:
                continue

            item = normalize_record(url, warc_date, html)
            if item:
                yield item


# -------------------------
# Build dataset (JSONL)
# -------------------------

def build_dataset(
    warc_files: List[str],
    out_jsonl: str,
    max_items: Optional[int] = None,
) -> None:
    os.makedirs(os.path.dirname(out_jsonl) or ".", exist_ok=True)

    seen_urls = set()
    n = 0

    total_files = len(warc_files)
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for wf in tqdm(warc_files, desc="WARC files"):
            for item in iter_warc_records(wf):
                u = item.get("url")
                if not u or u in seen_urls:
                    continue
                seen_urls.add(u)

                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                n += 1

                if max_items and n >= max_items:
                    print(f"STOP max_items={max_items}")
                    print(f"OK wrote {n} items -> {out_jsonl}")
                    return

    print(f"OK wrote {n} items -> {out_jsonl}")


# -------------------------
# Optional: JSONL -> Parquet
# -------------------------

def jsonl_to_parquet(jsonl_path: str, parquet_path: str) -> None:
    import pandas as pd  # lazy import
    import pyarrow  # noqa: F401

    os.makedirs(os.path.dirname(parquet_path) or ".", exist_ok=True)
    df = pd.read_json(jsonl_path, lines=True)
    df.to_parquet(parquet_path, index=False)
    print("OK parquet:", parquet_path, "shape=", df.shape)


# -------------------------
# MAIN
# -------------------------

if __name__ == "__main__":
    # Mets ici le dossier où tu as téléchargé tes 5 WARC
    INPUT_DIR = "ccnews_warc_last5/2026_02"  # <-- adapte
    OUT_JSONL = "outputs/ccnews_dataset.jsonl"
    OUT_PARQUET = "outputs/ccnews_dataset.parquet"

    warc_files = sorted(
        os.path.join(INPUT_DIR, f)
        for f in os.listdir(INPUT_DIR)
        if f.endswith(".warc.gz")
    )

    print("WARC files:", len(warc_files))
    if not warc_files:
        raise RuntimeError(f"Aucun .warc.gz trouvé dans {INPUT_DIR}")

    # Pour tester vite, limite à 5000 articles
    build_dataset(warc_files, OUT_JSONL, max_items=5000)

    # Optionnel: conversion parquet
    # jsonl_to_parquet(OUT_JSONL, OUT_PARQUET)


WARC files: 5


WARC files:   0%|          | 0/5 [00:00<?, ?it/s]/var/folders/kv/vpmhbd6j4gz_x_zf1_6jf5940000gn/T/ipykernel_13934/1965259794.py:76: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")
WARC files:   0%|          | 0/5 [05:42<?, ?it/s]

STOP max_items=5000
OK wrote 5000 items -> outputs/ccnews_dataset.jsonl


In [ ]:
# c'est celui là à lancer 

import os
import io
import gzip
import json
from urllib.parse import urlparse
from datetime import datetime
from typing import Optional, Dict, Any, Iterable, List

from warcio.archiveiterator import ArchiveIterator
from tqdm import tqdm

import trafilatura
from bs4 import BeautifulSoup


# -------------------------
# Utils dates / extraction
# -------------------------

def safe_dt(s: Optional[str]) -> Optional[str]:
    """Convertit une date ISO-ish en 'YYYY-MM-DD HH:MM:SS' si possible."""
    if not s:
        return None
    try:
        s2 = s.strip().replace("Z", "")
        dt = datetime.fromisoformat(s2)
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return None


def pick(extracted: Any, key: str, default=None):
    """
    Lit un champ depuis dict ou objet Document (trafilatura).
    Compatible avec plusieurs versions de trafilatura.
    """
    if extracted is None:
        return default

    if isinstance(extracted, dict):
        return extracted.get(key, default)

    if hasattr(extracted, "as_dict"):
        try:
            d = extracted.as_dict()
            if isinstance(d, dict):
                return d.get(key, default)
        except Exception:
            pass

    return getattr(extracted, key, default)


def normalize_date_value(v: Any) -> Optional[str]:
    """Accepte str ou datetime, renvoie 'YYYY-MM-DD HH:MM:SS' ou None."""
    if v is None:
        return None
    if hasattr(v, "strftime"):  # datetime-like
        try:
            return v.strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return None
    if isinstance(v, str):
        return safe_dt(v)
    return None


# -------------------------
# Fallback meta tags
# -------------------------

def meta_fallback(html: str) -> Dict[str, Optional[str]]:
    """Fallback simple via meta tags og:*, twitter:* et <title>."""
    soup = BeautifulSoup(html, "lxml")

    def get_meta(*keys: str) -> Optional[str]:
        for k in keys:
            tag = soup.find("meta", attrs={"property": k}) or soup.find("meta", attrs={"name": k})
            if tag and tag.get("content"):
                return tag["content"].strip()
        return None

    title = get_meta("og:title", "twitter:title")
    desc = get_meta("og:description", "twitter:description", "description")
    img = get_meta("og:image", "twitter:image")

    if not title:
        t = soup.find("title")
        title = t.get_text(strip=True) if t else None

    return {"title": title, "description": desc, "image_url": img}


# -------------------------
# Normalisation d'un record
# -------------------------

def normalize_record(url: str, warc_date: Optional[str], html: str) -> Optional[Dict[str, Any]]:
    """
    Construit un dict {date, description, domain, image_url, text, title, url}.
    date = publication si trouvée, sinon WARC-Date (crawl).
    """
    parsed = urlparse(url)
    domain = parsed.netloc

    extracted = trafilatura.bare_extraction(
        html,
        url=url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    if not extracted:
        return None

    text = (pick(extracted, "text", "") or "").strip()
    title = pick(extracted, "title", None)
    description = pick(extracted, "description", None)
    image_url = pick(extracted, "image", None)
    pub_date = pick(extracted, "date", None)

    date_norm = normalize_date_value(pub_date) or safe_dt(warc_date)

    # Fallback meta si manquant
    if not title or not description or not image_url:
        fb = meta_fallback(html)
        title = title or fb["title"]
        description = description or fb["description"]
        image_url = image_url or fb["image_url"]

    # Fallback description: début du texte
    if not description and text:
        description = (text[:280] + "…") if len(text) > 280 else text

    # Filtre anti-bruit minimal
    if len(text) < 200:
        return None
    if not title:
        return None

    return {
        "date": date_norm,
        "description": description,
        "domain": domain,
        "image_url": image_url,
        "text": text,
        "title": title,
        "url": url,
    }


# -------------------------
# Lecture WARC
# -------------------------

def iter_warc_records(warc_gz_path: str) -> Iterable[Dict[str, Any]]:
    """Yield records normalisés depuis un fichier .warc.gz."""
    with gzip.open(warc_gz_path, "rb") as stream:
        for rec in ArchiveIterator(stream):
            if rec.rec_type != "response":
                continue

            url = rec.rec_headers.get_header("WARC-Target-URI")
            warc_date = rec.rec_headers.get_header("WARC-Date")  # date crawl

            if not url:
                continue

            # Filtrer HTML
            ctype = ""
            if rec.http_headers:
                ctype = rec.http_headers.get_header("Content-Type") or ""
            if "text/html" not in (ctype or "").lower():
                continue

            try:
                raw = rec.content_stream().read()
                html = raw.decode("utf-8", errors="ignore")
            except Exception:
                continue

            # Skip XML (RSS, sitemap) -> évite warnings et accélère
            if html.lstrip().startswith("<?xml"):
                continue

            item = normalize_record(url, warc_date, html)
            if item:
                yield item


# -------------------------
# Build dataset (JSONL) + progress
# -------------------------

def build_dataset(
    warc_files: List[str],
    out_jsonl: str,
    max_items: Optional[int] = None,
) -> None:
    os.makedirs(os.path.dirname(out_jsonl) or ".", exist_ok=True)

    seen_urls = set()
    n = 0

    p_files = tqdm(warc_files, desc="WARC files", unit="file")
    p_items = tqdm(total=max_items, desc="Items written", unit="item")

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for wf in p_files:
            p_files.set_postfix_str(os.path.basename(wf))

            for item in iter_warc_records(wf):
                u = item.get("url")
                if not u or u in seen_urls:
                    continue
                seen_urls.add(u)

                f.write(json.dumps(item, ensure_ascii=False) + "\n")
                n += 1
                p_items.update(1)

                if max_items and n >= max_items:
                    p_items.close()
                    p_files.close()
                    print(f"STOP max_items={max_items}")
                    print(f"OK wrote {n} items -> {out_jsonl}")
                    return

    p_items.close()
    p_files.close()
    print(f"OK wrote {n} items -> {out_jsonl}")


# -------------------------
# Optional: JSONL -> Parquet
# -------------------------

def jsonl_to_parquet(jsonl_path: str, parquet_path: str) -> None:
    import pandas as pd
    import pyarrow  # noqa: F401

    os.makedirs(os.path.dirname(parquet_path) or ".", exist_ok=True)
    df = pd.read_json(jsonl_path, lines=True)
    df.to_parquet(parquet_path, index=False)
    print("OK parquet:", parquet_path, "shape=", df.shape)


# -------------------------
# MAIN
# -------------------------

if __name__ == "__main__":
    # Mets ici le dossier où tu as téléchargé tes .warc.gz
    INPUT_DIR = "ccnews_warc_last5/2026_02"  # <-- adapte à ton dossier
    OUT_JSONL = "outputs/ccnews_dataset.jsonl"
    OUT_PARQUET = "outputs/ccnews_dataset.parquet"

    warc_files = sorted(
        os.path.join(INPUT_DIR, f)
        for f in os.listdir(INPUT_DIR)
        if f.endswith(".warc.gz")
    )

    print("WARC files found:", len(warc_files))
    if not warc_files:
        raise RuntimeError(f"Aucun .warc.gz trouvé dans {INPUT_DIR}")

    # Affiche tailles pour comprendre le temps
    print("File sizes:")
    for wf in warc_files:
        print(" -", os.path.basename(wf), f"{os.path.getsize(wf)/1e6:.1f} MB")

    # Pour tester vite: baisse max_items à 200
    build_dataset(warc_files, OUT_JSONL, max_items=1000000)

    # Optionnel:
    # jsonl_to_parquet(OUT_JSONL, OUT_PARQUET)


WARC files found: 5
File sizes:
 - CC-NEWS-20260211224450-06781.warc.gz 1072.7 MB
 - CC-NEWS-20260212003559-06782.warc.gz 1072.7 MB
 - CC-NEWS-20260212025433-06783.warc.gz 1072.7 MB
 - CC-NEWS-20260212051501-06784.warc.gz 1072.7 MB
 - CC-NEWS-20260212071821-06785.warc.gz 1072.7 MB


WARC files:   0%|          | 0/5 [00:00<?, ?file/s, CC-NEWS-20260211224450-06781.warc.gz]
















































































































































































































































































































































































































































































































































































































































































































































































































































































































































KeyboardInterrupt: 

In [8]:
import pandas as pd

df = pd.read_json("outputs/ccnews_dataset.jsonl", lines=True)

# Top sources
print(df["domain"].value_counts().head(30))

# Presence d'un éditeur
print("washingtonpost.com count =", (df["domain"].str.contains("washingtonpost.com", na=False)).sum())


domain
menafn.com                     269
www.95047.it                   250
news.livedoor.com              170
www.hindustantimes.com         140
coinstrail.com                 128
www.law360.com                  82
www.wradio.com.co               78
www.elleadore.com               72
www.infobae.com                 71
timesofindia.indiatimes.com     67
g1.globo.com                    62
www.aydinses.com                61
ria.ru                          59
www.alaraby.co.uk               50
www.boursorama.com              43
www.stheadline.com              38
www.globenewswire.com           38
udn.com                         38
www.tuttomercatoweb.com         36
www.finanznachrichten.de        34
www.themarketsdaily.com         33
www.lanacion.com.ar             32
www.thehansindia.com            31
www.excelsior.com.mx            29
www.in.gr                       28
beninwebtv.com                  28
www.publimetro.co               27
www.cbssports.com               26
www.zazoom.it

In [ ]:
import os
import re
import gzip
import json
import zlib
from urllib.parse import urlparse
from datetime import datetime
from typing import Optional, Dict, Any, Iterable, List

from warcio.archiveiterator import ArchiveIterator
from tqdm import tqdm

import trafilatura
from bs4 import BeautifulSoup

# Détection d'encodage robuste (mieux que chardet en général)
from charset_normalizer import from_bytes  # pip install charset-normalizer

# Brotli (souvent rencontré sur CC / sites modernes)
try:
    import brotli  # pip install brotli
except Exception:
    brotli = None


# -------------------------
# Dates / extraction
# -------------------------

def safe_dt(s: Optional[str]) -> Optional[str]:
    """Convertit une date ISO-ish en 'YYYY-MM-DD HH:MM:SS' si possible."""
    if not s:
        return None
    try:
        s2 = s.strip().replace("Z", "")
        dt = datetime.fromisoformat(s2)
        return dt.strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return None


def pick(extracted: Any, key: str, default=None):
    """
    Lit un champ depuis dict ou objet Document (trafilatura).
    Compatible avec plusieurs versions de trafilatura.
    """
    if extracted is None:
        return default

    if isinstance(extracted, dict):
        return extracted.get(key, default)

    if hasattr(extracted, "as_dict"):
        try:
            d = extracted.as_dict()
            if isinstance(d, dict):
                return d.get(key, default)
        except Exception:
            pass

    return getattr(extracted, key, default)


def normalize_date_value(v: Any) -> Optional[str]:
    """Accepte str ou datetime, renvoie 'YYYY-MM-DD HH:MM:SS' ou None."""
    if v is None:
        return None
    if hasattr(v, "strftime"):  # datetime-like
        try:
            return v.strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            return None
    if isinstance(v, str):
        return safe_dt(v)
    return None


# -------------------------
# Décodage HTTP robuste
# -------------------------

CTRL_CHARS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")


def extract_charset(content_type: str) -> Optional[str]:
    """Parse 'charset=...' depuis Content-Type."""
    if not content_type:
        return None
    m = re.search(r"charset\s*=\s*([^\s;]+)", content_type, flags=re.I)
    if not m:
        return None
    cs = m.group(1).strip().strip('"').strip("'")
    return cs or None


def decode_http_body(raw: bytes, content_encoding: str, content_type: str) -> Optional[str]:
    """
    1) nettoie bytes
    2) décompresse selon Content-Encoding (br/gzip/deflate)
    3) décode selon charset header si présent, sinon auto-detect
    4) enlève caractères de contrôle
    """
    if not raw:
        return None

    # Certains contenus contiennent des NUL qui font planter des parseurs
    raw = raw.replace(b"\x00", b"")

    # Décompression
    enc = (content_encoding or "").lower()
    try:
        if "br" in enc:
            if brotli is None:
                return None
            raw = brotli.decompress(raw)
        elif "gzip" in enc:
            raw = gzip.decompress(raw)
        elif "deflate" in enc:
            raw = zlib.decompress(raw)
    except Exception:
        return None

    # Décodage via charset si présent, sinon auto
    charset = extract_charset(content_type or "")
    text: Optional[str] = None

    if charset:
        try:
            text = raw.decode(charset, errors="replace")
        except Exception:
            text = None

    if not text:
        try:
            best = from_bytes(raw).best()
            text = str(best) if best else raw.decode("utf-8", errors="replace")
        except Exception:
            return None

    # Retire caractères de contrôle (cause fréquente de "symboles" bizarres / crash)
    text = CTRL_CHARS_RE.sub("", text)
    return text


def sanitize_str(s: Optional[str]) -> Optional[str]:
    """Force une unicode 'safe' (remplace ce qui est invalide)."""
    if s is None:
        return None
    try:
        return s.encode("utf-8", "replace").decode("utf-8", "replace")
    except Exception:
        return None


def sanitize_url(u: Optional[str]) -> Optional[str]:
    if not u:
        return None
    u = u.replace("\x00", "").strip()
    return u or None


# -------------------------
# Fallback meta tags
# -------------------------

def meta_fallback(html: str) -> Dict[str, Optional[str]]:
    """Fallback simple via meta tags og:*, twitter:* et <title>."""
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    def get_meta(*keys: str) -> Optional[str]:
        for k in keys:
            tag = soup.find("meta", attrs={"property": k}) or soup.find("meta", attrs={"name": k})
            if tag and tag.get("content"):
                return tag["content"].strip()
        return None

    title = get_meta("og:title", "twitter:title")
    desc = get_meta("og:description", "twitter:description", "description")
    img = get_meta("og:image", "twitter:image")

    if not title:
        t = soup.find("title")
        title = t.get_text(strip=True) if t else None

    return {"title": title, "description": desc, "image_url": img}


# -------------------------
# Normalisation d'un record
# -------------------------

def normalize_record(url: str, warc_date: Optional[str], html: str) -> Optional[Dict[str, Any]]:
    """
    Construit un dict {date, description, domain, image_url, text, title, url}.
    date = publication si trouvée, sinon WARC-Date (crawl).
    """
    url = sanitize_url(url) or ""
    parsed = urlparse(url)
    domain = parsed.netloc or ""

    # Extraction texte/metadata via trafilatura
    try:
        extracted = trafilatura.bare_extraction(
            html,
            url=url,
            include_comments=False,
            include_tables=False,
            favor_precision=True,
        )
    except Exception:
        return None

    if not extracted:
        return None

    text = (pick(extracted, "text", "") or "").strip()
    title = pick(extracted, "title", None)
    description = pick(extracted, "description", None)
    image_url = pick(extracted, "image", None)
    pub_date = pick(extracted, "date", None)

    date_norm = normalize_date_value(pub_date) or safe_dt(warc_date)

    # Fallback meta si manquant
    if not title or not description or not image_url:
        fb = meta_fallback(html)
        title = title or fb["title"]
        description = description or fb["description"]
        image_url = image_url or fb["image_url"]

    # Fallback description: début du texte
    if not description and text:
        description = (text[:280] + "…") if len(text) > 280 else text

    # Anti-bruit minimal
    if len(text) < 200:
        return None
    if not title:
        return None

    # Sanitize strings pour éviter crash JSON/Unicode
    text = sanitize_str(text) or ""
    title = sanitize_str(title) or ""
    description = sanitize_str(description)
    image_url = sanitize_str(image_url)
    domain = sanitize_str(domain) or ""
    url = sanitize_str(url) or ""

    return {
        "date": date_norm,
        "description": description,
        "domain": domain,
        "image_url": image_url,
        "text": text,
        "title": title,
        "url": url,
    }


# -------------------------
# Logging erreurs (optionnel mais très utile)
# -------------------------

def log_error(path: str, payload: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    try:
        with open(path, "a", encoding="utf-8") as g:
            g.write(json.dumps(payload, ensure_ascii=False) + "\n")
    except Exception:
        pass


# -------------------------
# Lecture WARC
# -------------------------

def iter_warc_records(warc_gz_path: str, errors_path: Optional[str] = None) -> Iterable[Dict[str, Any]]:
    """Yield records normalisés depuis un fichier .warc.gz."""
    with gzip.open(warc_gz_path, "rb") as stream:
        for rec in ArchiveIterator(stream):
            if rec.rec_type != "response":
                continue

            url = rec.rec_headers.get_header("WARC-Target-URI")
            warc_date = rec.rec_headers.get_header("WARC-Date")  # date crawl

            url = sanitize_url(url)
            if not url:
                continue

            # Lire headers HTTP
            content_type = ""
            content_encoding = ""
            if rec.http_headers:
                content_type = rec.http_headers.get_header("Content-Type") or ""
                content_encoding = rec.http_headers.get_header("Content-Encoding") or ""

            # Filtrer HTML (avant decode)
            if "text/html" not in (content_type or "").lower():
                continue

            # Lire payload brut
            try:
                raw = rec.content_stream().read()
            except Exception:
                if errors_path:
                    log_error(errors_path, {
                        "warc_file": os.path.basename(warc_gz_path),
                        "url": url,
                        "warc_date": warc_date,
                        "content_type": content_type,
                        "content_encoding": content_encoding,
                        "note": "read_body_failed",
                    })
                continue

            # Decoder / decompress robustement
            html = decode_http_body(raw, content_encoding, content_type)
            if not html:
                if errors_path:
                    log_error(errors_path, {
                        "warc_file": os.path.basename(warc_gz_path),
                        "url": url,
                        "warc_date": warc_date,
                        "content_type": content_type,
                        "content_encoding": content_encoding,
                        "note": "decode_http_body_failed",
                        "body_head_hex": raw[:200].hex() if raw else None,
                    })
                continue

            # Skip XML (RSS, sitemap)
            if html.lstrip().startswith("<?xml"):
                continue

            item = normalize_record(url, warc_date, html)
            if item:
                yield item


# -------------------------
# Build dataset (JSONL) + progress
# -------------------------

def build_dataset(
    warc_files: List[str],
    out_jsonl: str,
    max_items: Optional[int] = None,
    errors_path: str = "outputs/errors.jsonl",
) -> None:
    os.makedirs(os.path.dirname(out_jsonl) or ".", exist_ok=True)

    seen_urls = set()
    n = 0

    p_files = tqdm(warc_files, desc="WARC files", unit="file", dynamic_ncols=True)
    p_items = tqdm(total=max_items, desc="Items written", unit="item", dynamic_ncols=True)

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for wf in p_files:
            p_files.set_postfix_str(os.path.basename(wf))

            for item in iter_warc_records(wf, errors_path=errors_path):
                u = item.get("url")
                if not u or u in seen_urls:
                    continue
                seen_urls.add(u)

                try:
                    f.write(json.dumps(item, ensure_ascii=False) + "\n")
                except Exception:
                    log_error(errors_path, {
                        "warc_file": os.path.basename(wf),
                        "url": u,
                        "note": "json_write_failed",
                    })
                    continue

                n += 1
                p_items.update(1)

                if max_items and n >= max_items:
                    p_items.close()
                    p_files.close()
                    print(f"STOP max_items={max_items}")
                    print(f"OK wrote {n} items -> {out_jsonl}")
                    return

    p_items.close()
    p_files.close()
    print(f"OK wrote {n} items -> {out_jsonl}")
    print(f"Errors (if any) -> {errors_path}")


# -------------------------
# Optional: JSONL -> Parquet
# -------------------------

def jsonl_to_parquet(jsonl_path: str, parquet_path: str) -> None:
    import pandas as pd
    import pyarrow  # noqa: F401

    os.makedirs(os.path.dirname(parquet_path) or ".", exist_ok=True)
    df = pd.read_json(jsonl_path, lines=True)
    df.to_parquet(parquet_path, index=False)
    print("OK parquet:", parquet_path, "shape=", df.shape)


# -------------------------
# MAIN
# -------------------------

if __name__ == "__main__":
    # Mets ici le dossier où tu as téléchargé tes .warc.gz
    INPUT_DIR = "ccnews_warc_last5/2026_02"  # <-- adapte à ton dossier
    OUT_JSONL = "outputs/ccnews_dataset.jsonl"
    OUT_PARQUET = "outputs/ccnews_dataset.parquet"
    ERRORS_JSONL = "outputs/errors.jsonl"

    warc_files = sorted(
        os.path.join(INPUT_DIR, f)
        for f in os.listdir(INPUT_DIR)
        if f.endswith(".warc.gz")
    )

    print("WARC files found:", len(warc_files))
    if not warc_files:
        raise RuntimeError(f"Aucun .warc.gz trouvé dans {INPUT_DIR}")

    # Affiche tailles pour comprendre le temps
    print("File sizes:")
    for wf in warc_files:
        try:
            print(" -", os.path.basename(wf), f"{os.path.getsize(wf)/1e6:.1f} MB")
        except Exception:
            print(" -", os.path.basename(wf), "(size: ? )")

    # Pour tester vite: max_items=200
    build_dataset(warc_files, OUT_JSONL, max_items=1_000_000, errors_path=ERRORS_JSONL)

    # Optionnel:
    # jsonl_to_parquet(OUT_JSONL, OUT_PARQUET)

Items written:   8%|▊         | 381/5000 [22:25:33<271:52:42, 211.90s/item]


WARC files found: 5
File sizes:
 - CC-NEWS-20260211224450-06781.warc.gz 1072.7 MB
 - CC-NEWS-20260212003559-06782.warc.gz 1072.7 MB
 - CC-NEWS-20260212025433-06783.warc.gz 1072.7 MB
 - CC-NEWS-20260212051501-06784.warc.gz 1072.7 MB
 - CC-NEWS-20260212071821-06785.warc.gz 1072.7 MB


WARC files:  60%|██████    | 3/5 [1:28:58<59:19, 1779.64s/file, CC-NEWS-20260212051501-06784.warc.gz]  


KeyboardInterrupt: 